# Task 1 — Data Cleaning and Duplicate Detection
## Women's Clothing: Trends and Analysis in the Fashion Industry

**Pipeline position:** Raw dataset → cleaned interim dataset.

This notebook reads the original CSV from `data/raw/`, audits duplicate rows and column quality, removes only defensibly irrelevant columns, and saves a cleaning log. It deliberately avoids deleting columns just because they look like identifiers or text: those fields may be important for product and merchant analysis.

**Input:** `data/raw/womens_clothing.csv`  
**Output:** `data/interim/womens_clothing_cleaned.csv` and audit logs in `logs/`.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

ROOT = Path.cwd()
# If the notebook is launched from another directory, locate the project root.
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

RAW_PATH = ROOT / "data" / "raw" / "womens_clothing.csv"
INTERIM_PATH = ROOT / "data" / "interim" / "womens_clothing_cleaned.csv"
LOG_DIR = ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {RAW_PATH}\n"
        "Place your original CSV at data/raw/womens_clothing.csv."
    )

df = pd.read_csv(RAW_PATH, low_memory=False)
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (1573, 43)


,title,title_orig,price,retail_price,currency_buyer,units_sold,uses_ad_boosts,rating,rating_count,rating_five_count,...,merchant_rating_count,merchant_rating,merchant_id,merchant_has_profile_picture,merchant_profile_picture,product_url,product_picture,product_id,theme,crawl_month
0,2020 Summer Vintage Flamingo Print Pajamas Se...,2020 Summer Vintage Flamingo Print Pajamas Se...,16.00,14,EUR,100,0,3.76,54,26.0,...,568,4.128521,595097d6a26f6e070cb878d1,0,NaN,https://www.wish.com/c/5e9ae51d43d6a96e303acdb0,https://contestimg.wish.com/api/webimage/5e9ae...,5e9ae51d43d6a96e303acdb0,summer,2020-08
1,SSHOUSE Summer Casual Sleeveless Soirée Party ...,Women's Casual Summer Sleeveless Sexy Mini Dress,8.00,22,EUR,20000,1,3.45,6135,2269.0,...,17752,3.899673,56458aa03a698c35c9050988,0,NaN,https://www.wish.com/c/58940d436a0d3d5da4e95a38,https://contestimg.wish.com/api/webimage/58940...,58940d436a0d3d5da4e95a38,summer,2020-08
2,2020 Nouvelle Arrivée Femmes Printemps et Été ...,2020 New Arrival Women Spring and Summer Beach...,8.00,43,EUR,100,0,3.57,14,5.0,...,295,3.989831,5d464a1ffdf7bc44ee933c65,0,NaN,https://www.wish.com/c/5ea10e2c617580260d55310a,https://contestimg.wish.com/api/webimage/5ea10...,5ea10e2c617580260d55310a,summer,2020-08
3,Hot Summer Cool T-shirt pour les femmes Mode T...,Hot Summer Cool T Shirt for Women Fashion Tops...,8.00,8,EUR,5000,1,4.03,579,295.0,...,23832,4.020435,58cfdefdacb37b556efdff7c,0,NaN,https://www.wish.com/c/5cedf17ad1d44c52c59e4aca,https://contestimg.wish.com/api/webimage/5cedf...,5cedf17ad1d44c52c59e4aca,summer,2020-08
4,Femmes Shorts d'été à lacets taille élastique ...,Women Summer Shorts Lace Up Elastic Waistband ...,2.72,3,EUR,100,1,3.10,20,6.0,...,14482,4.001588,5ab3b592c3911a095ad5dadb,0,NaN,https://www.wish.com/c/5ebf5819ebac372b070b0e70,https://contestimg.wish.com/api/webimage/5ebf5...,5ebf5819ebac372b070b0e70,summer,2020-08


## 1. Initial data audit

Review shape, missingness, uniqueness, and duplicate records before changing anything.

In [2]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_count": df.nunique(dropna=True)
})
display(audit.sort_values(["missing_percent", "unique_count"], ascending=[False, True]))
exact_duplicate_count = int(df.duplicated().sum())
print(f"Exact duplicate rows: {exact_duplicate_count:,}")

,dtype,missing_count,missing_percent,unique_count
merchant_profile_picture,str,1347,85.63,125
has_urgency_banner,float64,1100,69.93,1
urgency_text,str,1100,69.93,2
rating_two_count,float64,45,2.86,262
rating_one_count,float64,45,2.86,330
rating_three_count,float64,45,2.86,384
rating_four_count,float64,45,2.86,440
rating_five_count,float64,45,2.86,605
product_color,str,41,2.61,101
origin_country,str,17,1.08,6


Exact duplicate rows: 34


## 2. Remove duplicate rows with an audit trail

Exact duplicates are removed first. A second, optional duplicate check uses `product_id` only when that field exists and has usable values. It is disabled by default because repeated product IDs may represent legitimate variations or separate crawls.

In [3]:
duplicate_log = {
    "rows_before": int(len(df)),
    "exact_duplicate_rows": exact_duplicate_count,
    "product_id_duplicate_removal_enabled": False,
    "product_id_duplicate_rows_detected": int(df["product_id"].duplicated().sum())
        if "product_id" in df.columns else None
}
df_clean = df.drop_duplicates(keep="first").copy()
duplicate_log["rows_after_exact_deduplication"] = int(len(df_clean))
duplicate_log["rows_removed"] = int(len(df) - len(df_clean))

# Optional: enable only after confirming that one row per product_id is correct
# for your analysis. This can discard legitimate product variations.
REMOVE_DUPLICATE_PRODUCT_IDS = False
if REMOVE_DUPLICATE_PRODUCT_IDS and "product_id" in df_clean.columns:
    before = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=["product_id"], keep="first")
    duplicate_log["product_id_rows_removed"] = int(before - len(df_clean))

with open(LOG_DIR / "task1_duplicate_log.json", "w", encoding="utf-8") as f:
    json.dump(duplicate_log, f, indent=4)
print(duplicate_log)

{'rows_before': 1573, 'exact_duplicate_rows': 34, 'product_id_duplicate_removal_enabled': False, 'product_id_duplicate_rows_detected': 232, 'rows_after_exact_deduplication': 1539, 'rows_removed': 34}


## 3. Detect irrelevant or redundant columns conservatively

Automatic removal is limited to columns that are entirely empty or constant. Potential identifier, URL, image, and descriptive columns are retained because they may be useful for traceability, joins, or later analysis. Review the candidate report before changing the policy.

In [4]:
all_missing_cols = [c for c in df_clean.columns if df_clean[c].isna().all()]
constant_cols = [
    c for c in df_clean.columns
    if df_clean[c].nunique(dropna=True) <= 1 and c not in all_missing_cols
]
candidate_cols = sorted(set(all_missing_cols + constant_cols))

# Explicitly list columns to retain even if constant, if they are meaningful metadata.
PROTECTED_COLUMNS = {"product_id", "merchant_id", "currency_buyer", "origin_country"}
automatic_drop_cols = [c for c in candidate_cols if c not in PROTECTED_COLUMNS]

column_review = pd.DataFrame({
    "column": df_clean.columns,
    "missing_percent": (df_clean.isna().mean() * 100).round(2).values,
    "unique_count": df_clean.nunique(dropna=True).values,
    "all_missing": [c in all_missing_cols for c in df_clean.columns],
    "constant": [c in constant_cols for c in df_clean.columns],
    "automatic_removal_candidate": [c in automatic_drop_cols for c in df_clean.columns]
})
display(column_review.sort_values(
    ["automatic_removal_candidate", "missing_percent"],
    ascending=[False, False]
))

df_clean.drop(columns=automatic_drop_cols, inplace=True, errors="ignore")
column_log = {
    "all_missing_columns": all_missing_cols,
    "constant_columns": constant_cols,
    "automatically_removed_columns": automatic_drop_cols,
    "protected_columns": sorted(PROTECTED_COLUMNS),
    "columns_before": int(len(df.columns)),
    "columns_after": int(len(df_clean.columns))
}
with open(LOG_DIR / "task1_column_log.json", "w", encoding="utf-8") as f:
    json.dump(column_log, f, indent=4)
print("Removed columns:", automatic_drop_cols)

,column,missing_percent,unique_count,all_missing,constant,automatic_removal_candidate
27,has_urgency_banner,69.59,1,False,True,True
41,theme,0.00,1,False,True,True
42,crawl_month,0.00,1,False,True,True
37,merchant_profile_picture,85.38,125,False,False,False
28,urgency_text,69.59,2,False,False,False
9,rating_five_count,2.79,605,False,False,False
10,rating_four_count,2.79,440,False,False,False
11,rating_three_count,2.79,384,False,False,False
12,rating_two_count,2.79,262,False,False,False
13,rating_one_count,2.79,330,False,False,False


Removed columns: ['crawl_month', 'has_urgency_banner', 'theme']


C:\Users\Yatharth\AppData\Local\Temp\ipykernel_4880\951728632.py:25: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_clean.drop(columns=automatic_drop_cols, inplace=True, errors="ignore")


## 4. Save the cleaned dataset and summary

This interim CSV is the input to Task 2. No rows are removed for missing values in this task; missingness is recorded for later decisions.

In [5]:
INTERIM_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(INTERIM_PATH, index=False)

summary = {
    "input_file": str(RAW_PATH.relative_to(ROOT)),
    "output_file": str(INTERIM_PATH.relative_to(ROOT)),
    "rows_before": int(len(df)),
    "rows_after": int(len(df_clean)),
    "rows_removed": int(len(df) - len(df_clean)),
    "columns_before": int(len(df.columns)),
    "columns_after": int(len(df_clean.columns)),
    "columns_removed": automatic_drop_cols
}
with open(LOG_DIR / "task1_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print("Task 1 complete.")
print("Saved:", INTERIM_PATH)
print("Final shape:", df_clean.shape)
display(df_clean.head())

Task 1 complete.
Saved: d:\OneDrive\Desktop\Brainybeams\Womens_Clothing_ML_Project\data\interim\womens_clothing_cleaned.csv
Final shape: (1539, 40)


,title,title_orig,price,retail_price,currency_buyer,units_sold,uses_ad_boosts,rating,rating_count,rating_five_count,...,merchant_name,merchant_info_subtitle,merchant_rating_count,merchant_rating,merchant_id,merchant_has_profile_picture,merchant_profile_picture,product_url,product_picture,product_id
0,2020 Summer Vintage Flamingo Print Pajamas Se...,2020 Summer Vintage Flamingo Print Pajamas Se...,16.00,14,EUR,100,0,3.76,54,26.0,...,zgrdejia,(568 notes),568,4.128521,595097d6a26f6e070cb878d1,0,NaN,https://www.wish.com/c/5e9ae51d43d6a96e303acdb0,https://contestimg.wish.com/api/webimage/5e9ae...,5e9ae51d43d6a96e303acdb0
1,SSHOUSE Summer Casual Sleeveless Soirée Party ...,Women's Casual Summer Sleeveless Sexy Mini Dress,8.00,22,EUR,20000,1,3.45,6135,2269.0,...,sarahouse,"83 % avis positifs (17,752 notes)",17752,3.899673,56458aa03a698c35c9050988,0,NaN,https://www.wish.com/c/58940d436a0d3d5da4e95a38,https://contestimg.wish.com/api/webimage/58940...,58940d436a0d3d5da4e95a38
2,2020 Nouvelle Arrivée Femmes Printemps et Été ...,2020 New Arrival Women Spring and Summer Beach...,8.00,43,EUR,100,0,3.57,14,5.0,...,hxt520,86 % avis positifs (295 notes),295,3.989831,5d464a1ffdf7bc44ee933c65,0,NaN,https://www.wish.com/c/5ea10e2c617580260d55310a,https://contestimg.wish.com/api/webimage/5ea10...,5ea10e2c617580260d55310a
3,Hot Summer Cool T-shirt pour les femmes Mode T...,Hot Summer Cool T Shirt for Women Fashion Tops...,8.00,8,EUR,5000,1,4.03,579,295.0,...,allenfan,"(23,832 notes)",23832,4.020435,58cfdefdacb37b556efdff7c,0,NaN,https://www.wish.com/c/5cedf17ad1d44c52c59e4aca,https://contestimg.wish.com/api/webimage/5cedf...,5cedf17ad1d44c52c59e4aca
4,Femmes Shorts d'été à lacets taille élastique ...,Women Summer Shorts Lace Up Elastic Waistband ...,2.72,3,EUR,100,1,3.10,20,6.0,...,happyhorses,"85 % avis positifs (14,482 notes)",14482,4.001588,5ab3b592c3911a095ad5dadb,0,NaN,https://www.wish.com/c/5ebf5819ebac372b070b0e70,https://contestimg.wish.com/api/webimage/5ebf5...,5ebf5819ebac372b070b0e70


### Task 1 outcome
- Exact duplicate rows are removed and counted.
- Entirely empty and constant columns are identified and logged.
- Potentially useful identifiers and descriptive fields are preserved.
- The cleaned dataset is saved for Task 2.